# Identifying Most Zeroed Layers in OPT-125M Using Wanda Pruning

This notebook applies the Wanda pruning method to the OPT-125M model to identify transformer layers with the highest number of zeroed parameters, indicating high sparsity. Wanda combines weight magnitudes and activation statistics to prune insignificant parameters, and the resulting sparsity is analyzed to rank layers by their zero counts. The goal is to pinpoint layers that contribute least to model performance, which could be candidates for removal to optimize the model.

## Prerequisites
- Python 3.11.11
- GPU with CUDA support (e.g., NVIDIA Tesla T4)
- Required libraries: `transformers`, `datasets`, `accelerate`, `torch`, `numpy`, `matplotlib`, `bitsandbytes`, `peft`, `loralib`

## Notebook Structure
1. **Environment Setup**: Install dependencies and configure the runtime.
2. **Model and Data Preparation**: Load OPT-125M, tokenizer, and WikiText-2 dataset.
3. **Activation Collection**: Capture layer activations during inference.
4. **Wanda Pruning**: Prune the model to induce sparsity.
5. **Sparsity Analysis**: Rank layers by zeroed parameters and visualize results.
6. **Layer Selection**: Identify the most zeroed layers for potential removal.
7. **Model Export**: Save the pruned model and tokenizer.

## 1. Install Required Dependencies

Install Python packages necessary for model handling, dataset processing, pruning, and visualization. This ensures all tools are available to prune the model and analyze zeroed parameters.

In [ ]:
!pip install transformers datasets accelerate
!pip install -q bitsandbytes datasets accelerate loralib
!pip uninstall -y peft transformers
!pip install -q peft transformers

## 2. Import Libraries and Configure Device

Import libraries for model manipulation, data processing, and sparsity analysis. Configure the device to use GPU (CUDA) if available, otherwise default to CPU.

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from collections import defaultdict
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 3. Load Model and Tokenizer

Load the OPT-125M model and its tokenizer from the Hugging Face model hub. The model is set to automatic precision and moved to the selected device for efficient computation.

In [ ]:
model_name = "facebook/opt-125m"
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype='auto').to(device)
tokenizer = AutoTokenizer.from_pretrained(model_name)

## 4. Load and Tokenize Dataset

Load a 5% subset of the WikiText-2 validation dataset to collect activation statistics. Tokenize the text into batches with padding and truncation for model compatibility.

In [ ]:
# Load a small portion of WikiText-2 validation set
dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="validation[:5%]")

# Tokenize into batches
def tokenize(batch):
    return tokenizer(batch["text"], return_tensors="pt", padding=True, truncation=True, max_length=128)

tokenized = dataset.map(tokenize, batched=True)

## 5. Configure Activation Hooks

Set up hooks to capture input activations for all linear layers during forward passes. These activations are stored for use in the Wanda pruning process to evaluate parameter importance.

In [ ]:
activation_store = {}

def get_activation_hook(name):
    def hook(module, input, output):
        if isinstance(input[0], torch.Tensor):
            activation_store[name].append(input[0].detach().cpu())
    return hook

def register_hooks_recursively(module, prefix=''):
    for name, child in module.named_children():
        full_name = f'{prefix}.{name}' if prefix else name
        print(full_name)
        if isinstance(child, nn.Linear):
            activation_store[full_name] = []
            child.register_forward_hook(get_activation_hook(full_name))
        register_hooks_recursively(child, full_name)

register_hooks_recursively(model)

## 6. Collect Activations

Process the tokenized dataset in batches to collect activations. The model runs in evaluation mode without gradient computation to efficiently gather activation data for pruning.

In [ ]:
from transformers import default_data_collator

# Limit to first N samples
num_samples = 100
batch_size = 8
samples = tokenized.select(range(min(len(tokenized), num_samples)))

# Convert to list of dicts with only tensorizable fields
sample_dicts = []
for item in samples:
    item_tensors = {}
    for k in ['input_ids', 'attention_mask']:
        if k in item:
            item_tensors[k] = torch.tensor(item[k])
    sample_dicts.append(item_tensors)

# Run batches and collect activations
print(len(sample_dicts)/batch_size)

model.eval()
with torch.no_grad():
    for i in range(0, len(sample_dicts), batch_size):
        batch_items = sample_dicts[i:i+batch_size]
        print(f"sample {i}")
        # Use default collator to pad and batch
        batch = default_data_collator(batch_items)
        batch = {k: v.to(device) for k, v in batch.items()}

        model(**batch)

## 7. Apply Wanda Pruning

Apply the Wanda pruning algorithm, which zeros out parameters based on a score derived from weight magnitudes and mean activations. This step induces sparsity, enabling identification of layers with the most zeroed parameters.

In [ ]:
def wanda_prune_recursive(model, activations, threshold=1e-3, prefix=''):
    zero_counts = {}
    total_counts = {}

    for name, module in model.named_children():
        full_name = f"{prefix}.{name}" if prefix else name

        # Recurse into children
        child_zero, child_total = wanda_prune_recursive(module, activations, threshold, full_name)
        zero_counts.update(child_zero)
        total_counts.update(child_total)

        # If it's a Linear layer, apply pruning
        if isinstance(module, nn.Linear) and full_name in activations:
            print(full_name)
            W = module.weight.data.cpu()
            A = torch.cat(activations[full_name], dim=0)

            A_mean = A.mean(dim=0)
            print(A_mean.shape)
            print(W.shape)

            # Handle case for 1D or 2D A_mean
            if A_mean.dim() == 1:  # A_mean is 1D (e.g., for fully connected layers)
                scores = torch.abs(W * A_mean[None, :])  # [out_features, in_features]
            elif A_mean.dim() == 2:  # A_mean is 2D (e.g., for attention layers)
                scores = torch.abs(W * A_mean.mean(dim=0)[None, :])  # [out_features, in_features]
            else:
                continue

            mask = scores < threshold
            W[mask] = 0.0
            module.weight.data = W.to(module.weight.device)

            zero_counts[full_name] = mask.sum().item()
            total_counts[full_name] = W.numel()

    return zero_counts, total_counts

zero_counts, total_counts = wanda_prune_recursive(model, activation_store, threshold=1e-3)

## 8. Rank Layers by Zero Counts

Calculate the number of zeroed parameters for each linear layer (excluding the language model head). Sort layers by zero counts to identify those with the highest sparsity and visualize the sparsity ratios.

In [ ]:
del total_counts['lm_head']
del zero_counts['lm_head']

layer_zero_stats = [
    (name, zero_counts[name], total_counts[name], zero_counts[name] / total_counts[name])
    for name in zero_counts
]

layer_zero_stats.sort(key=lambda x: x[1], reverse=True)

print(f"{'Layer':50} | {'Zeros':>10} | {'Total':>10} | {'Sparsity':>10}")
print("-" * 80)
for name, zeros, total, ratio in layer_zero_stats[:12]:
    print(f"{name:50} | {zeros:10} | {total:10} | {ratio:9.2%}")

import matplotlib.pyplot as plt

names = [name for name, _, _, _ in layer_zero_stats]
sparsities = [s for _, _, _, s in layer_zero_stats]

plt.figure(figsize=(24, 12))
plt.barh(names[::-1], sparsities[::-1])
plt.xlabel("Sparsity Ratio")
plt.title("Layer-wise Sparsity for Identifying Most Zeroed Layers")
plt.tight_layout()
plt.savefig('zeroed_layers_sparsity.png')

## 9. Identify Most Zeroed Transformer Layers

Aggregate sparsity statistics by transformer layer and sort by sparsity ratio. Select the top layers with the highest number of zeroed parameters as candidates for removal.

In [ ]:
layer_stats = []
for i in range(12):
    k = {'name': f"layer {i}", 'zeros': 0, 'total': 0, 'sparsity': 0}
    for j in range(len(layer_zero_stats)):
        if f".{i}." in layer_zero_stats[j][0]:
            k['zeros'] += layer_zero_stats[j][1]
            k['total'] += layer_zero_stats[j][2]
            k['sparsity'] = k['zeros'] / k['total']
    layer_stats.append(k)

sorted_data = sorted(layer_stats, key=lambda x: x['sparsity'], reverse=True)

for item in sorted_data:
    broadcasting item

Number_of_layers_to_delete = 3
Layers_to_delete = [int(name['name'].split(' ')[1]) for name in sorted_data[:Number_of_layers_to_delete]]
print(f"Layers to delete: {Layers_to_delete}")

## 10. Save Pruned Model

Save the pruned OPT-125M model and tokenizer to a directory for further analysis or deployment, preserving the sparsity structure.

In [ ]:
model.save_pretrained("wanda-pruned-opt-125m")
tokenizer.save_pretrained("wanda-pruned-opt-125m")